# Camada Silver: canais

Trata a tabela `bronze.canais` e grava `silver.tb_canais`. Padroniza o nome e o tipo de cada canal a partir de dicionários de grafias e valida que nenhum campo fica nulo.

In [0]:
from pyspark.sql import functions as F, Window

catalog = "vcredit_medallion"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

## Funções auxiliares

In [0]:
ACENTOS = "áàâãäéèêëíìîïóòôõöúùûüç"
SEM_ACENTO = "aaaaaeeeeiiiiooooouuuuc"


def chave(c):
    """Texto em minúsculas, sem acento, sem espaços nas pontas e com espaços internos únicos.
    Serve só para comparar grafias; o valor final vem do dicionário de mapeamento."""
    t = F.lower(F.trim(F.col(c).cast("string")))
    return F.regexp_replace(F.translate(t, ACENTOS, SEM_ACENTO), r"\s+", " ")


def mapear(c, mapa):
    """Troca o valor da coluna `c` pelo valor do dicionário. Valor fora do dicionário vira null
    (e as validações abaixo barram o notebook se isso acontecer)."""
    expr = None
    for k, v in mapa.items():
        expr = F.when(F.col(c) == k, v) if expr is None else expr.when(F.col(c) == k, v)
    return expr


def ultima_versao(df, chaves):
    """Fica com a carga mais recente de cada chave (a Bronze é append) e descarta ingestion_datetime."""
    w = Window.partitionBy(*chaves).orderBy(F.col("ingestion_datetime").desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn", "ingestion_datetime")


def checar(regra, df_falhas):
    """Para o notebook se alguma linha violar a regra."""
    n = df_falhas.count()
    assert n == 0, f"FALHA na validação '{regra}': {n} linha(s)"
    print(f"OK: {regra}")


def salvar(df, nome):
    (df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{silver_schema}.{nome}"))
    print(f"{silver_schema}.{nome}: {spark.table(f'{silver_schema}.{nome}').count()} linhas")

## 1. Canais

In [0]:
# Inclui as variações de grafia vistas também em chamados.canal
MAPA_CANAL = {
    "ura": "URA", "u.r.a": "URA",
    "chatbot": "Chatbot", "chat bot": "Chatbot",
    "atendimento inicial": "Atendimento Inicial", "atendimento_inicial": "Atendimento Inicial",
    "atend. inicial": "Atendimento Inicial",
    "atendimento especializado": "Atendimento Especializado", "atendimento_especializado": "Atendimento Especializado",
    "atend. especializado": "Atendimento Especializado",
}
MAPA_TIPO_CANAL = {"autosservico": "Autosserviço", "humano": "Humano"}

b_canais = spark.table(f"{bronze_schema}.canais")
display(b_canais)

In [0]:
canais = ultima_versao(
    b_canais.withColumn("id_canal", F.trim(F.col("id_canal")).cast("int")),
    ["id_canal"],
)
canais = (
    canais
    .withColumn("k_nome", chave("nome_canal"))
    .withColumn("k_tipo", chave("tipo_canal"))
    .withColumn("nome_canal", mapear("k_nome", MAPA_CANAL))
    .withColumn("tipo_canal", mapear("k_tipo", MAPA_TIPO_CANAL))
    .withColumn("status_canal", F.lower(F.trim(F.col("status_canal"))))
    .select("id_canal", "nome_canal", "tipo_canal", "status_canal")
    .orderBy("id_canal")
)

checar("canais: nenhum campo nulo", canais.filter(
    F.col("id_canal").isNull() | F.col("nome_canal").isNull()
    | F.col("tipo_canal").isNull() | F.col("status_canal").isNull()))
checar("canais: id_canal único", canais.groupBy("id_canal").count().filter("count > 1"))
checar("canais: status é ativo ou inativo", canais.filter(~F.col("status_canal").isin("ativo", "inativo")))
display(canais)

## 2. Gravação da Silver
Só chega aqui se todas as validações passaram.

In [0]:
salvar(canais, "tb_canais")